### Build A Basic Chatbot With Langgraph (Graph Api)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

In [ ]:
class State(TypedDict):
    # add messages is reducer that append mess rather then overriding
    messages: Annotated[list, add_messages]
graph_builder = StateGraph(State)
graph_builder

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


In [ ]:
from langchain_groq import ChatGroq
agent = ChatGroq(
    model="llama-3.3-70b-versatile",
)

In [ ]:
def chatbot(state: State):
    return {"messages":[agent.invoke(state["messages"])]}

In [ ]:
graph_builder.add_node("chatbot",chatbot)
graph_builder.add_edge(START,"chatbot")
graph_builder.add_edge("chatbot",END)

# compile the graph
graph = graph_builder.compile()

In [ ]:
from IPython.display import display,Image
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
response = graph.invoke({
    "messages": [HumanMessage(content="hi")]
})

print(response)

In [ ]:
for event in graph.stream({"messages":"Hi how are you"}):
    for value in event.values():
        print(value["messages"][-1].content)

### ChatBot With Tools

In [ ]:
from langchain_tavily import TavilySearch
tool = TavilySearch(max_results=2)
tool.invoke("what is langraph")

In [ ]:
## Custom function
def multiply(a:int,b:int)->int:
    """Multiply a and b

    Args:
        a (int): first int
        b (int): second int

    Returns:
        int: output int
    """
    return a*b
tools=[tool,multiply]
llm_with_tools = agent.bind_tools(tools)

In [ ]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode,tools_condition

# Node def
def tool_calling_llm(state:State):
    return {"messages":llm_with_tools.invoke(state["messages"])}

# Add nodes
graph_builder = StateGraph(State)
graph_builder.add_node("tool_calling_llm",tool_calling_llm)
graph_builder.add_node("tools",ToolNode(tools))

## Add Edges
graph_builder.add_edge(START, "tool_calling_llm")
graph_builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition
)
graph_builder.add_edge("tools",END)

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
response = graph.invoke({"messages":[HumanMessage(content="what is recent ai news and then multiply 124 by 432")]})
response["messages"][-1].content


In [ ]:
for m in response["messages"]:
    m.pretty_print()

### ReAct Agent Architecture

In [ ]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode,tools_condition

# Node def
def tool_calling_llm(state:State):
    return {"messages":llm_with_tools.invoke(state["messages"])}

# Add nodes
graph_builder = StateGraph(State)
graph_builder.add_node("tool_calling_llm",tool_calling_llm)
graph_builder.add_node("tools",ToolNode(tools))

## Add Edges
graph_builder.add_edge(START, "tool_calling_llm")
graph_builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition
)
graph_builder.add_edge("tools","tool_calling_llm")

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
for m in response["messages"]:
    m.pretty_print()

### Adding memory in agentic graph

In [ ]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode,tools_condition
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
# Node def
def tool_calling_llm(state:State):
    return {"messages":llm_with_tools.invoke(state["messages"])}

# Add nodes
graph_builder = StateGraph(State)
graph_builder.add_node("tool_calling_llm",tool_calling_llm)
graph_builder.add_node("tools",ToolNode(tools))

## Add Edges
graph_builder.add_edge(START, "tool_calling_llm")
graph_builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition
)
graph_builder.add_edge("tools","tool_calling_llm")

graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
config={"configurable":{"thread_id":"1"}}
response = graph.invoke({"messages":"Hi my name is Sharjeel"},config=config)
response

In [ ]:
response['messages'][-1].content


In [ ]:
response = graph.invoke({"messages":"Hi what is my name"},config=config)
response['messages'][-1].content


### Streaming

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

def superbot(state:State):
    return {"messages":agent.invoke(state["messages"])}

In [ ]:
graph=StateGraph(State)

## node
graph.add_node("SuperBot",superbot)
## Edges

graph.add_edge(START,"SuperBot")
graph.add_edge("SuperBot",END)


graph_builder=graph.compile(checkpointer=memory)


## Display
from IPython.display import Image, display
display(Image(graph_builder.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "1"}}
graph_builder.invoke({'messages':"Hi,My name is Sharjeel And I like cricket"},config)

In [ ]:
# Create a thread
config = {"configurable": {"thread_id": "3"}}

for chunk in graph_builder.stream({'messages':"Hi,My name is Sharjeel And I like cricket"},config,stream_mode="updates"):
    print(chunk)

In [ ]:
for chunk in graph_builder.stream({'messages':"Hi,My name is sharjeel And I like cricket"},config,stream_mode="values"):
    print(chunk)

In [ ]:
config = {"configurable": {"thread_id": "5"}}

async for event in graph_builder.astream_events({"messages":["Hi My name is Krish and I like to play cricket"]},config,version="v2"):
    print(event)